In [ ]:
# Celula unica - Importacao de dados GPKG para PostgreSQL
import geopandas as gpd
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

# =============
# CONFIGURACOES 
# =============

# Pasta onde estao os arquivos GPKG
PASTA_DADOS = Path(r"C:\Temp")

# Lista de arquivos CAR
CAR_ARQUIVOS = [
    "es_pi_car_20260406.gpkg"
]

# Arquivo SIGEF
SIGEF_ARQUIVO = "pa_br_sigef_privado_20260107.gpkg"

# Controle de importacao
IMPORTAR_CAR = True
IMPORTAR_SIGEF = False              #ATIVAR COM "True" CASO AINDA NÃO TENHA IMPORTADO A TABELA NA AULA ANTERIOR!

# Credenciais do banco local
DB_HOST = "localhost"
DB_PORT = "5432"
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_NAME = "geotec"

# ==========================================
# FUNCOES
# ==========================================

def conectar_banco():
    engine = create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')
    return engine

def criar_schemas(engine):
    with engine.connect() as conn:
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS car"))
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS incra"))
        conn.execute(text("CREATE EXTENSION IF NOT EXISTS postgis"))
        conn.commit()
    print("[OK] Schemas car e incra verificados/criados")

def verificar_tabela(engine, schema, tabela):
    try:
        with engine.connect() as conn:
            conn.execute(text(f"SELECT 1 FROM {schema}.{tabela} LIMIT 1"))
            return True
    except:
        return False

def contar_registros(engine, schema, tabela):
    try:
        with engine.connect() as conn:
            result = conn.execute(text(f"SELECT COUNT(*) FROM {schema}.{tabela}"))
            return result.fetchone()[0]
    except:
        return 0

def importar_gpkg_geometria(engine, caminho_arquivo, schema, tabela):
    """Importa arquivos com geometria (CAR, SIGEF)"""
    print(f"  Lendo arquivo: {caminho_arquivo.name}")
    gdf = gpd.read_file(str(caminho_arquivo))
    print(f"  Registros encontrados: {len(gdf)}")
    
    # Garante que a coluna geometry está no formato correto
    if 'geometry' in gdf.columns:
        gdf = gdf.set_geometry('geometry')
    
    # Força o CRS para SIRGAS 2000 (EPSG:4674)
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4674)
    else:
        gdf = gdf.to_crs(epsg=4674)
    
    # Importa para o PostgreSQL
    gdf.to_postgis(
        name=tabela,
        con=engine,
        schema=schema,
        if_exists='replace',
        index=False
    )
    print(f"  Importado para {schema}.{tabela}")
    
    return len(gdf)
    
    # Importa para o PostgreSQL usando to_sql (pandas)
    df.to_sql(
        name=tabela,
        con=engine,
        schema=schema,
        if_exists='replace',
        index=False
    )
    print(f"  Importado para {schema}.{tabela}")
    
    return len(df)

# ==========================================
# MAIN
# ==========================================

def main():
    print("=" * 50)
    print("IMPORTACAO DE DADOS GEOPROCESSAMENTO")
    print("=" * 50)
    
    # Verificar pasta
    if not PASTA_DADOS.exists():
        print(f"\n[ERRO] Pasta nao encontrada: {PASTA_DADOS}")
        return
    
    arquivos_faltando = []

    # Verificar CARs
    if IMPORTAR_CAR:
        for arquivo in CAR_ARQUIVOS:
            caminho = PASTA_DADOS / arquivo
            if not caminho.exists():
                arquivos_faltando.append(arquivo)

    # Verificar SIGEF
    sigef_path = PASTA_DADOS / SIGEF_ARQUIVO
    if IMPORTAR_SIGEF and not sigef_path.exists():
        arquivos_faltando.append(SIGEF_ARQUIVO)

    if arquivos_faltando:
        print(f"\n[ERRO] Arquivos nao encontrados: {arquivos_faltando}")
        return
    
    # Conexao
    print("\n[1] Conectando ao PostgreSQL...")
    try:
        engine = conectar_banco()
        print("[OK] Conexao estabelecida")
    except Exception as e:
        print(f"[ERRO] {e}")
        return
    
    # Schemas
    print("\n[2] Configurando schemas...")
    criar_schemas(engine)
    
    # =====================
    # IMPORTAR CAR
    # =====================
    if IMPORTAR_CAR:
        print("\n[3] Importando CAR...")
        
        for arquivo in CAR_ARQUIVOS:
            caminho = PASTA_DADOS / arquivo
            tabela = Path(arquivo).stem
            
            print(f"\n--> {arquivo}")
            
            if verificar_tabela(engine, "car", tabela):
                registros = contar_registros(engine, "car", tabela)
                print(f"[INFO] car.{tabela} ja existe ({registros} registros)")
                
                resposta = input("[?] Recriar? (s/N): ")
                if resposta.lower() == 's':
                    importar_gpkg_geometria(engine, caminho, "car", tabela)
                else:
                    print("  Mantido")
            else:
                importar_gpkg_geometria(engine, caminho, "car", tabela)

    else:
        print("\n[3] Importacao do CAR desativada")

    # =====================
    # IMPORTAR SIGEF
    # =====================
    if IMPORTAR_SIGEF:
        print("\n[4] Importando SIGEF...")
        
        tabela = Path(SIGEF_ARQUIVO).stem
        
        if verificar_tabela(engine, "incra", tabela):
            registros = contar_registros(engine, "incra", tabela)
            print(f"[INFO] incra.{tabela} ja existe ({registros} registros)")
            
            resposta = input("[?] Recriar? (s/N): ")
            if resposta.lower() == 's':
                importar_gpkg_geometria(engine, sigef_path, "incra", tabela)
            else:
                print("  Mantido")
        else:
            importar_gpkg_geometria(engine, sigef_path, "incra", tabela)

    else:
        print("\n[4] Importacao do SIGEF desativada")

    # =====================
    # RESUMO FINAL
    # =====================
    if IMPORTAR_CAR:
        print("\nResumo CAR:")
        for arquivo in CAR_ARQUIVOS:
            tabela = Path(arquivo).stem
            registros = contar_registros(engine, "car", tabela)
            print(f"  {tabela}: {registros} registros")

    if IMPORTAR_SIGEF:
        tabela = Path(SIGEF_ARQUIVO).stem
        registros = contar_registros(engine, "incra", tabela)
        print(f"\nSIGEF: {registros} registros")

    engine.dispose()
    
    print("\n" + "=" * 50)
    print("PRONTO! AGORA PODEMOS SEGUIR COM O CURSO")
    print("=" * 50)

if __name__ == "__main__":
    main()